# 50a — Sonic blowdown: the design and a crude CFD smoke test

**The design.** A 1 L supply container at 50 bar (absolute) and 20 °C feeds a 10 mm bore pipe, 100 mm long. A ball
valve sits 20 mm down the pipe. It opens at t = 0 and stays open for the 10 ms simulated. The pipe blows into still
air at 1 atm. Two gases: air and CO₂.

**The smoke test.** The full OpenFOAM case (`rhoCentralFoam`, axisymmetric wedge, k-ω SST) on a deliberately
**thick mesh**: 1.25 mm cells in the bore (only 4 across the radius), about 2,000 cells in all. It runs in minutes and
checks the whole chain: CAD → case → mesh → solver → function objects → plots. The numbers are approximate: the
boundary layer, the vena contracta at the pipe entrance and the shocks in the jet are all under-resolved. The
production run (0.5 mm cells, notebook 50) gives the numbers to quote.

Code: `sonic_flow.py` (CAD, case writer, runner, 0-D model).

In [ ]:
DEBUG = False                                  # real OpenFOAM (no mocks)
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)
import math
from dataclasses import asdict
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
import sonic_flow as sf, sonic_flow_report as rep
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.color": "#e4e3df", "axes.spines.top": False,
                     "axes.spines.right": False, "lines.linewidth": 2.0, "legend.frameon": False})
COL = {"air": "#2a78d6", "co2": "#eb6834"}; LAB = {"air": "Air", "co2": "CO$_2$"}
rig, cond = sf.Rig(), sf.Conditions()
OUT = Path("output/sonic_smoke"); RES = Path("sonic_flow_results")

## 1. The design

In [ ]:
print(f"pipe bore {rig.pipe_bore} mm, length {rig.pipe_length} mm, exit area {rig.exit_area_m2*1e6:.2f} mm²")
print(f"valve seat at x = {rig.valve_x} mm from the container")
print(f"container {rig.container_volume_l} L: bore {rig.container_bore} mm, length {rig.container_length:.1f} mm")
print(f"container {cond.p0/1e5:.0f} bar, {cond.T0} K; ambient {cond.p_amb/1e5:.5f} bar; flow time {cond.end_time*1e3:.0f} ms")
for g in sf.GASES.values():
    print(f"{g.name:4s} M = {g.mol_weight} kg/kmol, cp = {g.cp} J/kg K, gamma = {g.gamma:.3f}, "
          f"critical pressure ratio p*/p0 = {(2/(g.gamma+1))**(g.gamma/(g.gamma-1)):.3f}  (50 bar -> 1 atm: choked)")

In [ ]:
cad = sf.make_cad(RES / "cad", rig)                   # Dedalus / CadQuery -> STEP + STL
print(f"fluid volume: {cad['fluid_volume_l']:.4f} L measured on the CAD, {cad['fluid_volume_expected_l']:.4f} L expected")
rep.plot_cad(RES / "cad", RES)
display(Image(RES / "cad_supply_rig.png"), Image(RES / "cad_fluid_volume.png"))

The CFD solves the gas only: the (x, r) profile below, revolved through a 5° wedge, plus the still air beyond the
exit. The valve is a diaphragm at x = 20 mm. Everything upstream of it starts at 50 bar.

In [ ]:
mesh = sf.Mesh(pipe=1.25, container_far=10.0, ambient_far=10.0)     # crude: 4 cells across the bore radius
fig, ax = plt.subplots(figsize=(10, 3.6))
for x0, x1, r0, r1, nx, nr, ex, er, zone in mesh.blocks(rig):
    ax.add_patch(plt.Rectangle((x0, r0), x1 - x0, r1 - r0, fill=True, alpha=0.25,
                 color={"container": "#2a78d6", "pipeUpstream": "#eb6834", "pipeDownstream": "#1baf7a", "ambient": "#b0aea6"}[zone]))
    ax.text((x0 + x1) / 2, (r0 + r1) / 2, f"{zone}\n{nx}×{nr}", ha="center", va="center", fontsize=7)
ax.axvline(rig.valve_x, color="k", ls="--", lw=1); ax.text(rig.valve_x + 1, 45, "valve", fontsize=8)
ax.set_xlim(-rig.container_length - 5, rig.pipe_length + rig.ambient_length + 5); ax.set_ylim(0, 102)
ax.set_xlabel("x [mm]"); ax.set_ylabel("r [mm]"); ax.set_title("Mesh blocks (cells x × r) — crude smoke-test mesh", loc="left")
plt.show()

## 2. Crude OpenFOAM run, both gases (10 ms each, a few minutes)

In [ ]:
import time
cases = {}
for g in ("air", "co2"):
    t0 = time.time()
    cases[g] = sf.write_case(OUT / g, sf.GASES[g], rig, sf.Conditions(write_interval=1e-3), mesh)
    sf.run_case(cases[g])
    cells = [l for l in (cases[g] / "log.checkMesh").read_text().splitlines() if "cells:" in l][0].split()[-1]
    print(f"{g}: {cells} cells, dt = {sf.time_step(sf.GASES[g], cond, mesh):.2e} s, wall time {time.time() - t0:.0f} s")

In [ ]:
H = {g: rep.clean(sf.histories(c)) for g, c in cases.items()}
Z = {g: (sf.blowdown(sf.GASES[g], rig, cond), sf.blowdown(sf.GASES[g], rig, cond, real=True, n=80)) for g in H}
for g, h in H.items():
    s = sf.summary(h, sf.GASES[g])
    print(f"{g:4s} exit velocity: peak {s['exit_velocity_peak_m_s']:.0f}, at 10 ms {s['exit_velocity_end_m_s']:.0f} m/s | "
          f"exit Mach at 10 ms {s['exit_mach_end']:.2f} | mass flow at 10 ms {s['mass_flow_end_kg_s']:.3f} kg/s | "
          f"released {s['released_mass_g']:.1f} g | container {s['container_pressure_end_bar']:.1f} bar, {s['container_temperature_end_K']:.0f} K")

## 3. Results
### Exit gas velocity vs time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for g, h in H.items():
    ax.plot(h["t"] * 1e3, h["exit_U"], color=COL[g], label=f"{LAB[g]} — CFD (crude mesh)")
    ideal, real = Z[g]
    ax.plot(ideal["t"] * 1e3, ideal["U"], color=COL[g], ls="--", lw=1.5, label=f"{LAB[g]} — 0-D ideal gas, sonic throat")
    ax.plot(real["t"] * 1e3, real["U"], color=COL[g], ls=":", label=f"{LAB[g]} — 0-D real gas (CoolProp)")
ax.set_xlabel("time after the valve opens [ms]"); ax.set_ylabel("exit velocity (mass-averaged) [m/s]")
ax.set_title("Exit gas velocity — 10 mm pipe, 50 bar, valve open 0–10 ms", loc="left"); ax.set_xlim(0, 10); ax.set_ylim(bottom=0)
ax.legend(fontsize=8); plt.show()

### Other parameters

In [ ]:
panels = [("exit_Ma", None, "Exit Mach number", "–", 1), ("exit_mdot", "mdot", "Mass flow at the exit", "kg/s", 1),
          ("exit_p", "p_star", "Exit static pressure (0-D: p*)", "bar", 1e-5), ("exit_T_mass", "T_star", "Exit temperature (0-D: T*)", "K", 1),
          ("container_p", "p", "Container pressure", "bar", 1e-5), ("container_T", "T", "Container temperature", "K", 1),
          ("released_mass", "released_mass", "Gas released", "g", 1e3), ("valve_mdot", "mdot", "Mass flow through the valve", "kg/s", 1)]
fig, axes = plt.subplots(4, 2, figsize=(12, 13), sharex=True)
for ax, (key, zk, title, unit, k) in zip(axes.flat, panels):
    for g, h in H.items():
        ax.plot(h["t"] * 1e3, h[key] * k, color=COL[g], label=f"{LAB[g]} CFD")
        if zk:
            ax.plot(Z[g][0]["t"] * 1e3, Z[g][0][zk] * k, color=COL[g], ls="--", lw=1.4, label=f"{LAB[g]} 0-D ideal")
            ax.plot(Z[g][1]["t"] * 1e3, Z[g][1][zk] * k, color=COL[g], ls=":", label=f"{LAB[g]} 0-D real")
    if key == "exit_Ma":
        ax.axhline(1, color="#52514e", ls="-.", lw=1)
    ax.set_title(title, loc="left"); ax.set_ylabel(unit)
for ax in axes[-1]:
    ax.set_xlabel("time [ms]")
axes[0][1].legend(fontsize=7.5, ncol=2); fig.tight_layout(); plt.show()

### Mach number in the pipe and the jet

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 7.5))
xe = rig.pipe_length * 1e-3
for j, g in enumerate(H):
    for i, tm in enumerate((1e-3, 5e-3, 10e-3)):
        ax = axes[i][j]
        x, r, ma = rep.mach_slice(cases[g], min(rep._times(cases[g]), key=lambda v: abs(v - tm)))
        k = (x > xe - 0.06) & (x < xe + 0.12) & (r < 0.03)
        sc = ax.scatter(np.r_[x[k], x[k]] * 1e3, np.r_[r[k], -r[k]] * 1e3, c=np.r_[ma[k], ma[k]], s=9, marker="s",
                        cmap="magma", vmin=0, vmax=3)
        ax.axvline(rig.pipe_length, color="w", lw=0.8, ls="--")
        ax.set_title(f"{LAB[g]}, t = {tm*1e3:.0f} ms (cell centres)", loc="left", fontsize=9); ax.set_aspect("equal")
        ax.grid(False); ax.set_ylabel("r [mm]")
    axes[-1][j].set_xlabel("x [mm]  (pipe exit at 100, dashed)")
fig.colorbar(sc, ax=axes, shrink=0.7, label="Mach number"); plt.show()

## 4. What the smoke test says

- **The chain works**: CAD → wedge mesh → `setFields` (valve closed) → `rhoCentralFoam` → function objects → histories.
- **The pipe chokes within ~0.2 ms**: the start-up shock reaches the exit, then the exit runs at or above Mach 1 for the
  whole 10 ms. With a 50 : 1 pressure ratio the exit never becomes subsonic in this window.
- **Exit velocity**: the 0-D sonic throat gives a* (air ≈ 313 m/s, ideal CO₂ ≈ 250 m/s). The CFD exit velocity runs
  **above** it, because the sharp pipe entrance makes a vena contracta. That contraction is a throat, and the gas
  re-expands supersonically down the bore until friction brings it back towards Mach 1. The crude mesh resolves
  neither the contraction nor the wall friction well, so treat its levels as ±20 %.
- **Container**: it empties and cools isentropically, about 50 → 40 bar and 293 → 275 K in 10 ms (air). The 0-D curves
  bracket the CFD.
- **CO₂** releases more mass at a lower velocity (heavier, γ = 1.29). The real-gas model (dotted) shows why the ideal
  CO₂ CFD is only indicative: at 50 bar CO₂ is denser than an ideal gas (Z ≈ 0.64). Its expansion to the throat crosses
  the saturation line, so about 8 % condenses (homogeneous equilibrium) and the choked velocity falls to ~190 m/s.

Next: the production mesh (0.5 mm, notebook 50) for the numbers to quote.